In [2]:
pip install tqdm

  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
Using cached tqdm-4.67.1-py3-none-any.whl (78 kB)
Note: you may need to restart the kernel to use updated packages.


In [3]:
# ============================================================
# Early Fusion: Digital + Thermal (Input-Level Fusion)
# ============================================================
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv2D, MaxPooling2D, Concatenate,
                                     GlobalAveragePooling2D, Dense, Dropout,
                                     BatchNormalization, Activation)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.applications import ResNet50
import numpy as np
import os
from glob import glob
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

# ---------------------- CONFIG ----------------------
digital_train_dir = r"C:\Users\cl502_15\Downloads\Mtech Major Project\Mtech Major Project\Dataset\Maturity\train test val split for digital\train"
thermal_train_dir = r"C:\Users\cl502_15\Downloads\Mtech Major Project\Mtech Major Project\Dataset\Maturity\train test val split  thermal\train"
digital_val_dir = r"C:\Users\cl502_15\Downloads\Mtech Major Project\Mtech Major Project\Dataset\Maturity\train test val split for digital\val"
thermal_val_dir = r"C:\Users\cl502_15\Downloads\Mtech Major Project\Mtech Major Project\Dataset\Maturity\train test val split  thermal\val"

BATCH_SIZE = 16
IMAGE_SIZE = (224, 224)
EPOCHS = 50
LR = 1e-4

# ============================================================
# STEP 1: Load and combine images at input level
# ============================================================
def load_combined_images(digital_dir, thermal_dir):
    """
    Load digital and thermal images and stack them as 6-channel input
    (3 RGB channels + 3 thermal channels)
    """
    print(f"\nLoading images from {os.path.basename(digital_dir)}...")
    
    classes = sorted(os.listdir(digital_dir))
    class_to_idx = {cls: idx for idx, cls in enumerate(classes)}
    
    combined_images = []
    labels = []
    
    for class_name in classes:
        digital_class_dir = os.path.join(digital_dir, class_name)
        thermal_class_dir = os.path.join(thermal_dir, class_name)
        
        if not os.path.isdir(digital_class_dir) or not os.path.isdir(thermal_class_dir):
            continue
        
        digital_files = sorted(glob(os.path.join(digital_class_dir, '*.*')))
        thermal_files = sorted(glob(os.path.join(thermal_class_dir, '*.*')))
        
        print(f"  Class '{class_name}': {len(digital_files)} digital, {len(thermal_files)} thermal")
        
        min_files = min(len(digital_files), len(thermal_files))
        
        for i in range(min_files):
            # Load digital image (RGB - 3 channels)
            img_d = load_img(digital_files[i], target_size=IMAGE_SIZE)
            img_d = img_to_array(img_d) / 255.0
            
            # Load thermal image (RGB - 3 channels)
            img_t = load_img(thermal_files[i], target_size=IMAGE_SIZE)
            img_t = img_to_array(img_t) / 255.0
            
            # Concatenate along channel axis: (224, 224, 3) + (224, 224, 3) = (224, 224, 6)
            combined = np.concatenate([img_d, img_t], axis=-1)
            combined_images.append(combined)
            
            labels.append(class_to_idx[class_name])
    
    combined_images = np.array(combined_images, dtype='float32')
    labels = np.array(labels)
    
    print(f"  Total loaded: {len(labels)} combined images")
    print(f"  Combined shape: {combined_images.shape}")
    
    return combined_images, labels, len(classes)

print("="*60)
print("EARLY FUSION: Loading combined images")
print("="*60)

# Load training data
X_train_combined, y_train, NUM_CLASSES = load_combined_images(
    digital_train_dir, thermal_train_dir
)

# Load validation data
X_val_combined, y_val, _ = load_combined_images(
    digital_val_dir, thermal_val_dir
)

# Convert labels to categorical
y_train_cat = to_categorical(y_train, NUM_CLASSES)
y_val_cat = to_categorical(y_val, NUM_CLASSES)

print(f"\nDataset summary:")
print(f"  Training samples: {len(y_train)}")
print(f"  Validation samples: {len(y_val)}")
print(f"  Number of classes: {NUM_CLASSES}")
print(f"  Input shape: {X_train_combined.shape[1:]}")

# ============================================================
# STEP 2: Build Early Fusion Model
# ============================================================
print("\n" + "="*60)
print("Building Early Fusion Model")
print("="*60)

def build_early_fusion_cnn(input_shape, num_classes):
    """
    Custom CNN that takes 6-channel input (3 RGB + 3 Thermal)
    """
    inputs = Input(shape=input_shape, name='combined_input')
    
    # Block 1
    x = Conv2D(64, (3, 3), padding='same', name='conv1_1')(inputs)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Conv2D(64, (3, 3), padding='same', name='conv1_2')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = MaxPooling2D((2, 2), name='pool1')(x)
    
    # Block 2
    x = Conv2D(128, (3, 3), padding='same', name='conv2_1')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Conv2D(128, (3, 3), padding='same', name='conv2_2')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = MaxPooling2D((2, 2), name='pool2')(x)
    
    # Block 3
    x = Conv2D(256, (3, 3), padding='same', name='conv3_1')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Conv2D(256, (3, 3), padding='same', name='conv3_2')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Conv2D(256, (3, 3), padding='same', name='conv3_3')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = MaxPooling2D((2, 2), name='pool3')(x)
    
    # Block 4
    x = Conv2D(512, (3, 3), padding='same', name='conv4_1')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Conv2D(512, (3, 3), padding='same', name='conv4_2')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Conv2D(512, (3, 3), padding='same', name='conv4_3')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = MaxPooling2D((2, 2), name='pool4')(x)
    
    # Block 5
    x = Conv2D(512, (3, 3), padding='same', name='conv5_1')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Conv2D(512, (3, 3), padding='same', name='conv5_2')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Conv2D(512, (3, 3), padding='same', name='conv5_3')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = MaxPooling2D((2, 2), name='pool5')(x)
    
    # Global pooling
    x = GlobalAveragePooling2D(name='global_pool')(x)
    
    # Classification head
    x = Dense(512, activation='relu', name='fc1')(x)
    x = Dropout(0.5)(x)
    x = Dense(256, activation='relu', name='fc2')(x)
    x = Dropout(0.4)(x)
    
    outputs = Dense(num_classes, activation='softmax', name='predictions')(x)
    
    model = Model(inputs=inputs, outputs=outputs, name='early_fusion_cnn')
    return model

# Build model
input_shape = X_train_combined.shape[1:]  # (224, 224, 6)
early_fusion_model = build_early_fusion_cnn(input_shape, NUM_CLASSES)

# Compile
early_fusion_model.compile(
    optimizer=Adam(learning_rate=LR),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("\nModel Architecture:")
early_fusion_model.summary()

# ============================================================
# STEP 3: Alternative - Using Pretrained Model with Modified Input
# ============================================================
print("\n" + "="*60)
print("Alternative: ResNet50-based Early Fusion")
print("="*60)

def build_early_fusion_resnet(input_shape, num_classes):
    """
    Use ResNet50 with modified input layer to accept 6 channels
    """
    # Create custom input
    inputs = Input(shape=input_shape, name='combined_input')
    
    # Add 1x1 conv to map 6 channels to 3 channels for ResNet
    x = Conv2D(3, (1, 1), padding='same', name='channel_adapter')(inputs)
    
    # Load ResNet50 without top layer (no input_tensor to avoid layer mismatch)
    base_model = ResNet50(
        include_top=False,
        weights='imagenet',
        input_shape=(224, 224, 3),
        pooling='avg'
    )
    
    # Pass through ResNet
    x = base_model(x, training=False)
    
    # Freeze base model initially
    base_model.trainable = False
    
    # Add classification head
    x = Dense(512, activation='relu', name='fc1')(x)
    x = Dropout(0.5)(x)
    x = Dense(256, activation='relu', name='fc2')(x)
    x = Dropout(0.4)(x)
    outputs = Dense(num_classes, activation='softmax', name='predictions')(x)
    
    model = Model(inputs=inputs, outputs=outputs, name='early_fusion_resnet50')
    return model

# Build alternative model
early_fusion_resnet = build_early_fusion_resnet(input_shape, NUM_CLASSES)

# Compile
early_fusion_resnet.compile(
    optimizer=Adam(learning_rate=LR),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("\nResNet50-based Early Fusion Architecture:")
early_fusion_resnet.summary()

# ============================================================
# STEP 4: Choose which model to train
# ============================================================
print("\n" + "="*60)
print("Model Selection")
print("="*60)
print("\nTwo models available:")
print("  1. Custom CNN (trains from scratch)")
print("  2. ResNet50-based (uses pretrained weights)")
print("\nSelect model to train:")
print("  Enter 1 for Custom CNN")
print("  Enter 2 for ResNet50 (recommended)")

# Default to custom CNN to avoid weight loading issues
model_choice = 2  # Change to 2 if you want ResNet50

if model_choice == 1:
    print("\nTraining Custom CNN from scratch...")
    model_to_train = early_fusion_model
    model_name = "early_fusion_cnn"
else:
    print("\nTraining ResNet50-based model...")
    model_to_train = early_fusion_resnet
    model_name = "early_fusion_resnet50"

# ============================================================
# STEP 5: Setup callbacks
# ============================================================
os.makedirs('models', exist_ok=True)

callbacks = [
    ModelCheckpoint(
        f'models/best_{model_name}.h5',
        monitor='val_accuracy',
        mode='max',
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_accuracy',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    )
]

# ============================================================
# STEP 6: Train model
# ============================================================
print("\n" + "="*60)
print("Training Early Fusion Model")
print("="*60)

history = model_to_train.fit(
    X_train_combined,
    y_train_cat,
    validation_data=(X_val_combined, y_val_cat),
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

# ============================================================
# STEP 7: Save model
# ============================================================
model_to_train.save(f'models/final_{model_name}.h5')
print(f"\nModel saved to 'models/final_{model_name}.h5'")

# Save history
import pickle
with open(f'models/{model_name}_history.pkl', 'wb') as f:
    pickle.dump(history.history, f)

# ============================================================
# STEP 8: Evaluation and Classification Report
# ============================================================
print("\n" + "="*60)
print("EVALUATION ON VALIDATION SET")
print("="*60)

# Make predictions
y_pred_probs = model_to_train.predict(X_val_combined, batch_size=BATCH_SIZE)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = y_val

# Get class names
class_names = sorted(os.listdir(digital_train_dir))
print(f"\nClass names: {class_names}")

# Classification report
print("\nClassification Report:")
print("="*60)
report = classification_report(y_true, y_pred, target_names=class_names, digits=4)
print(report)

# Save classification report
with open(f'models/{model_name}_classification_report.txt', 'w') as f:
    f.write("Early Fusion Classification Report\n")
    f.write("="*60 + "\n")
    f.write(report)

# Confusion Matrix
print("\nGenerating confusion matrix...")
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix - Early Fusion', fontsize=16, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.savefig(f'models/{model_name}_confusion_matrix.png', dpi=300, bbox_inches='tight')
print(f"Confusion matrix saved to 'models/{model_name}_confusion_matrix.png'")
plt.close()

# ============================================================
# STEP 9: Training History Visualization
# ============================================================
print("\nGenerating training visualizations...")

# Create figure with 2 subplots
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Accuracy
axes[0].plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
axes[0].set_title('Model Accuracy - Early Fusion', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].legend(loc='lower right', fontsize=10)
axes[0].grid(True, alpha=0.3)

# Plot 2: Loss
axes[1].plot(history.history['loss'], label='Training Loss', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[1].set_title('Model Loss - Early Fusion', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Loss', fontsize=12)
axes[1].legend(loc='upper right', fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'models/{model_name}_training_history.png', dpi=300, bbox_inches='tight')
print(f"Training history plot saved to 'models/{model_name}_training_history.png'")
plt.close()

# ============================================================
# STEP 10: Per-class Accuracy
# ============================================================
print("\nGenerating per-class accuracy visualization...")

per_class_accuracy = []
for i in range(NUM_CLASSES):
    class_mask = y_true == i
    if np.sum(class_mask) > 0:
        class_acc = np.sum((y_pred == y_true) & class_mask) / np.sum(class_mask)
        per_class_accuracy.append(class_acc)
    else:
        per_class_accuracy.append(0)

plt.figure(figsize=(10, 6))
bars = plt.bar(class_names, per_class_accuracy, color='coral', edgecolor='darkred', alpha=0.7)
plt.title('Per-Class Accuracy - Early Fusion', fontsize=14, fontweight='bold')
plt.xlabel('Class', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.ylim([0, 1.0])
plt.grid(axis='y', alpha=0.3)

for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.3f}',
             ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig(f'models/{model_name}_per_class_accuracy.png', dpi=300, bbox_inches='tight')
print(f"Per-class accuracy plot saved to 'models/{model_name}_per_class_accuracy.png'")
plt.close()

# ============================================================
# STEP 11: Final Summary
# ============================================================
print("\n" + "="*60)
print("FINAL SUMMARY - EARLY FUSION")
print("="*60)

final_train_acc = history.history['accuracy'][-1]
final_val_acc = history.history['val_accuracy'][-1]
best_val_acc = max(history.history['val_accuracy'])
best_epoch = history.history['val_accuracy'].index(best_val_acc) + 1

print(f"\nModel: {model_name}")
print(f"Input: 6-channel (3 RGB + 3 Thermal)")
print(f"\nFinal Training Accuracy:   {final_train_acc:.4f}")
print(f"Final Validation Accuracy: {final_val_acc:.4f}")
print(f"Best Validation Accuracy:  {best_val_acc:.4f} (Epoch {best_epoch})")

print(f"\nSaved files in 'models/' directory:")
print(f"  - best_{model_name}.h5")
print(f"  - final_{model_name}.h5")
print(f"  - {model_name}_history.pkl")
print(f"  - {model_name}_classification_report.txt")
print(f"  - {model_name}_confusion_matrix.png")
print(f"  - {model_name}_training_history.png")
print(f"  - {model_name}_per_class_accuracy.png")

print("\n" + "="*60)
print("Early Fusion training completed successfully!")
print("="*60)

EARLY FUSION: Loading combined images

Loading images from train...
  Class 'immature': 2280 digital, 2424 thermal
  Class 'mature': 2257 digital, 2346 thermal
  Class 'semi_mature': 1994 digital, 1869 thermal
  Total loaded: 6406 combined images
  Combined shape: (6406, 224, 224, 6)

Loading images from val...
  Class 'immature': 488 digital, 519 thermal
  Class 'mature': 483 digital, 502 thermal
  Class 'semi_mature': 427 digital, 400 thermal
  Total loaded: 1371 combined images
  Combined shape: (1371, 224, 224, 6)

Dataset summary:
  Training samples: 6406
  Validation samples: 1371
  Number of classes: 3
  Input shape: (224, 224, 6)

Building Early Fusion Model

Model Architecture:


Model: "early_fusion_cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ combined_input (InputLayer)     │ (None, 224, 224, 6)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1_1 (Conv2D)                │ (None, 224, 224, 64)   │         3,520 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_26          │ (None, 224, 224, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_26 (Activation)      │ (None, 224, 224, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1_2 (Conv2D)                │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_27          │ (None, 224, 224, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_27 (Activation)      │ (None, 224, 224, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool1 (MaxPooling2D)            │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2_1 (Conv2D)                │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_28          │ (None, 112, 112, 128)  │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_28 (Activation)      │ (None, 112, 112, 128)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2_2 (Conv2D)                │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_29          │ (None, 112, 112, 128)  │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_29 (Activation)      │ (None, 112, 112, 128)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool2 (MaxPooling2D)            │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3_1 (Conv2D)                │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_30          │ (None, 56, 56, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_30 (Activation)      │ (None, 56, 56, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3_2 (Conv2D)                │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_31          │ (None, 56, 56, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_31 (Activation)      │ (None, 56, 56, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3_3 (Conv2D)                │ (None, 56, 56, 256)    │       590,08

 Total params: 15,128,067 (57.71 MB)

 Trainable params: 15,119,619 (57.68 MB)

...

401/401 ━━━━━━━━━━━━━━━━━━━━ 494s 1s/step - accuracy: 0.5072 - loss: 0.9752 - val_accuracy: 0.5587 - val_loss: 0.9441 - learning_rate: 1.0000e-04
Epoch 42/50
401/401 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5124 - loss: 0.9757
Epoch 42: val_accuracy did not improve from 0.55872
401/401 ━━━━━━━━━━━━━━━━━━━━ 494s 1s/step - accuracy: 0.5014 - loss: 0.9793 - val_accuracy: 0.5295 - val_loss: 0.9521 - learning_rate: 1.0000e-04
Epoch 43/50
401/401 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5037 - loss: 0.9751
Epoch 43: val_accuracy improved from 0.55872 to 0.56236, saving model to models/best_early_fusion_resnet50.h5


401/401 ━━━━━━━━━━━━━━━━━━━━ 497s 1s/step - accuracy: 0.5095 - loss: 0.9702 - val_accuracy: 0.5624 - val_loss: 0.9336 - learning_rate: 1.0000e-04
Epoch 44/50
401/401 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5171 - loss: 0.9710
Epoch 44: val_accuracy did not improve from 0.56236
401/401 ━━━━━━━━━━━━━━━━━━━━ 494s 1s/step - accuracy: 0.5142 - loss: 0.9699 - val_accuracy: 0.5398 - val_loss: 0.9423 - learning_rate: 1.0000e-04
Epoch 45/50
401/401 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5168 - loss: 0.9659
Epoch 45: val_accuracy improved from 0.56236 to 0.56893, saving model to models/best_early_fusion_resnet50.h5


401/401 ━━━━━━━━━━━━━━━━━━━━ 496s 1s/step - accuracy: 0.5181 - loss: 0.9661 - val_accuracy: 0.5689 - val_loss: 0.9340 - learning_rate: 1.0000e-04
Epoch 46/50
401/401 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5085 - loss: 0.9656
Epoch 46: val_accuracy did not improve from 0.56893
401/401 ━━━━━━━━━━━━━━━━━━━━ 495s 1s/step - accuracy: 0.5133 - loss: 0.9651 - val_accuracy: 0.5631 - val_loss: 0.9246 - learning_rate: 1.0000e-04
Epoch 47/50
401/401 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5162 - loss: 0.9633
Epoch 47: val_accuracy improved from 0.56893 to 0.58425, saving model to models/best_early_fusion_resnet50.h5


401/401 ━━━━━━━━━━━━━━━━━━━━ 494s 1s/step - accuracy: 0.5165 - loss: 0.9662 - val_accuracy: 0.5842 - val_loss: 0.9179 - learning_rate: 1.0000e-04
Epoch 48/50
401/401 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5244 - loss: 0.9613
Epoch 48: val_accuracy did not improve from 0.58425
401/401 ━━━━━━━━━━━━━━━━━━━━ 494s 1s/step - accuracy: 0.5245 - loss: 0.9628 - val_accuracy: 0.5791 - val_loss: 0.9185 - learning_rate: 1.0000e-04
Epoch 49/50
401/401 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5232 - loss: 0.9598
Epoch 49: val_accuracy did not improve from 0.58425
401/401 ━━━━━━━━━━━━━━━━━━━━ 494s 1s/step - accuracy: 0.5250 - loss: 0.9593 - val_accuracy: 0.5842 - val_loss: 0.9195 - learning_rate: 1.0000e-04
Epoch 50/50
401/401 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5188 - loss: 0.9630
Epoch 50: val_accuracy did not improve from 0.58425
401/401 ━━━━━━━━━━━━━━━━━━━━ 498s 1s/step - accuracy: 0.5229 - loss: 0.9602 - val_accuracy: 0.5784 - val_loss: 0.9022 - learning_rate: 1.0000e-04
Re


Model saved to 'models/final_early_fusion_resnet50.h5'

EVALUATION ON VALIDATION SET
86/86 ━━━━━━━━━━━━━━━━━━━━ 50s 557ms/step

Class names: ['immature', 'mature', 'semi_mature']

Classification Report:
              precision    recall  f1-score   support

    immature     0.6710    0.5266    0.5901       488
      mature     0.5247    0.7267    0.6094       483
 semi_mature     0.6050    0.4825    0.5369       400

    accuracy                         0.5842      1371
   macro avg     0.6002    0.5786    0.5788      1371
weighted avg     0.6002    0.5842    0.5814      1371


Generating confusion matrix...
Confusion matrix saved to 'models/early_fusion_resnet50_confusion_matrix.png'

Generating training visualizations...
Training history plot saved to 'models/early_fusion_resnet50_training_history.png'

Generating per-class accuracy visualization...
Per-class accuracy plot saved to 'models/early_fusion_resnet50_per_class_accuracy.png'

FINAL SUMMARY - EARLY FUSION

Model: early_fusi